# 제주 특산물 가격 예측 - Keras DNN 모델링

| 항목 | 내용 |
|------|------|
| **버전** | v1.0.1 |
| **날짜** | 2026-03-12 |
| **모델** | Keras Deep Neural Network (DNN) |
| **전처리 참고** | DACON 1위 솔루션 전처리 반영 |
| **출력** | results/submission_v1.0.1.csv |

## 변경 이력
| 버전 | 날짜 | 내용 |
|------|------|------|
| v1.0.0 | 2026-03-12 | 초기 버전 - Keras DNN 기본 모델 (lag/rolling 피처) |
| v1.0.1 | 2026-03-12 | DACON 1위 전처리 반영 (holiday, year_month 누적, week_num, get_dummies, TG sqrt, 후처리) |

## v1.0.1 변경 내용
| 항목 | v1.0.0 | v1.0.1 |
|------|--------|--------|
| 전처리 방식 | train/test 따로 | **train+test 합쳐서** 처리 |
| 시간 피처 | quarter, season | **year_month(누적), week_num(누적)** 추가 |
| 공휴일 | ❌ | **holiday 변수** 추가 |
| 이상치 처리 | IQR 클리핑 | **품목별 임계값 → 평균 대체** |
| 범주형 인코딩 | ❌ | **item, corporation, location get_dummies** |
| TG 타겟 변환 | log1p (전체 동일) | **sqrt 변환** (TG 별도 처리) |
| 후처리 | ❌ | **품목별 최솟값 미만 → 0** |

---
## 1. 라이브러리 로드

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
import datetime
import random
import os
import platform
warnings.filterwarnings('ignore')

import holidays

# Keras / TensorFlow
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks, regularizers

# Sklearn
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 한글 폰트
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
else:
    plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

# 시드 고정
SEED = 2024
random.seed(SEED)
np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
tf.random.set_seed(SEED)

print(f'TensorFlow: {tf.__version__}')
print(f'GPU 사용: {len(tf.config.list_physical_devices("GPU")) > 0}')

---
## 2. 데이터 로드

In [ ]:
DATA_PATH = './data/'

train = pd.read_csv(DATA_PATH + 'train.csv', encoding='utf-8-sig')
test  = pd.read_csv(DATA_PATH + 'test.csv',  encoding='utf-8-sig')
sub   = pd.read_csv(DATA_PATH + 'sample_submission.csv', encoding='utf-8-sig')

print(f'train shape: {train.shape}')
print(f'test  shape: {test.shape}')
print(f'컬럼: {list(train.columns)}')
train.head(3)

---
## 3. 전처리 (DACON 1위 참고)

### 핵심: train + test 합쳐서 전처리
> 인코딩 일관성을 유지하고 테스트 데이터에도 동일한 변환 적용

In [ ]:
def pre_all(train, test):
    """
    DACON 1위 솔루션 참고 전처리 함수
    - train + test 합쳐서 처리 (인코딩 일관성)
    - 추가 피처: year_month(누적), week_num(누적), holiday
    """
    print(f'전처리 전 train: {train.shape}, test: {test.shape}')
    print('전처리 중...')

    train['timestamp'] = pd.to_datetime(train['timestamp'])
    test['timestamp']  = pd.to_datetime(test['timestamp'])

    # train + test 합치기
    df = pd.concat([train, test]).reset_index(drop=True)
    df.rename(columns={'supply(kg)': 'supply', 'price(원/kg)': 'price'}, inplace=True)

    # 기본 날짜 피처
    df['year']     = df['timestamp'].dt.year
    df['month']    = df['timestamp'].dt.month
    df['day']      = df['timestamp'].dt.day
    df['week_day'] = df['timestamp'].dt.weekday  # 0=월, 6=일

    # year_month: 누적 개월 수 (LabelEncoder)
    # → 2019-1=0, 2019-2=1, ..., 2023-3=50 형태로 시간 흐름 표현
    le = LabelEncoder()
    df['year_month'] = df['timestamp'].map(lambda x: str(x.year) + '-' + str(x.month))
    df['year_month'] = le.fit_transform(df['year_month'])

    # week: ISO 주차
    df['week'] = df['timestamp'].map(
        lambda x: datetime.datetime(x.year, x.month, x.day).isocalendar()[1]
    )

    # week_num: 연도별 누적 주차 (2019 기준 절대 주차)
    week_offsets = {2019: 0, 2020: 52, 2021: 52+53, 2022: 52+53+53, 2023: 52+53+53+52}
    df['week_num'] = df.apply(
        lambda r: int(r['week']) + week_offsets.get(r['year'], 0), axis=1
    )
    # 2019년 12월 마지막 주 보정 (isocalendar 버그)
    df.loc[df['timestamp'] == '2019-12-30', 'week_num'] = 52
    df.loc[df['timestamp'] == '2019-12-31', 'week_num'] = 52

    # holiday: 한국 공휴일 (0/1)
    kr_holi = holidays.KR()
    df['holiday'] = df['timestamp'].map(lambda x: 1 if x in kr_holi else 0)

    # train / test 분리
    train_out = df[~df['price'].isnull()].sort_values('timestamp').reset_index(drop=True)
    test_out  = df[df['price'].isnull()].sort_values('timestamp').reset_index(drop=True)

    print(f'전처리 후 train: {train_out.shape}, test: {test_out.shape}')
    return train_out, test_out


train_pre, test_pre = pre_all(train, test)
print(f'\n생성된 컬럼: {list(train_pre.columns)}')

In [ ]:
# -------------------------------------------------------
# 이상치 처리: 품목별 임계값 초과 → 해당 품목 평균으로 대체
# (DACON 1위 참고 - IQR 대신 도메인 지식 기반 임계값)
# -------------------------------------------------------
outlier_thresholds = {
    'TG': 20000,
    'RD': 5000,
    'BC': 8000,
    'CB': 2300,
}

for item, threshold in outlier_thresholds.items():
    idx = train_pre[(train_pre['item'] == item) & (train_pre['price'] > threshold)].index
    if len(idx) > 0:
        mean_price = train_pre[(train_pre['item'] == item) & (train_pre['price'] != 0)]['price'].mean()
        train_pre.loc[idx, 'price'] = mean_price
        print(f'{item}: {len(idx)}개 이상치 → 평균({mean_price:.0f})으로 대체')

print('\n이상치 처리 완료')

In [ ]:
# -------------------------------------------------------
# 범주형 인코딩: item, corporation, location → get_dummies
# supply 컬럼 제외 (테스트 데이터에 없음)
# -------------------------------------------------------
drop_cols_train = ['supply', 'timestamp']
drop_cols_test  = ['supply', 'timestamp', 'price']

Xy_all = pd.get_dummies(
    train_pre.drop(columns=drop_cols_train),
    columns=['item', 'corporation', 'location']
).sort_values('ID').reset_index(drop=True)

test_enc = pd.get_dummies(
    test_pre.drop(columns=drop_cols_test),
    columns=['item', 'corporation', 'location']
).sort_values('ID').reset_index(drop=True)

print(f'Xy_all 컬럼 ({len(Xy_all.columns)}개):')
print(list(Xy_all.columns))

---
## 4. TG / 비TG 분리 처리

> **TG(감귤)** 는 가격 범위가 다른 품목과 크게 달라 **sqrt 변환** 적용 (DACON 1위 참고)  
> 나머지 품목(BC, CB, CR, RD)은 **log1p 변환** 적용

In [ ]:
# TG 공휴일 보정: 공휴일이지만 실제 거래된 날은 holiday=0으로 수정
tg_mask = (train_pre['item'] == 'TG') & (train_pre['holiday'] == 1) & (train_pre['price'] != 0)
active_holi = list(
    train_pre[tg_mask].groupby('timestamp').count().reset_index()['timestamp']
)
fix_idx = train_pre[train_pre['timestamp'].isin(active_holi)].index
train_pre.loc[fix_idx, 'holiday'] = 0
# Xy_all도 동기화
Xy_all = pd.get_dummies(
    train_pre.drop(columns=drop_cols_train),
    columns=['item', 'corporation', 'location']
).sort_values('ID').reset_index(drop=True)
print(f'TG 공휴일 보정: {len(fix_idx)}개')

# 피처 컬럼 정의
ID_COL    = 'ID'
TARGET_COL = 'price'
FEAT_COLS = [c for c in Xy_all.columns if c not in [ID_COL, TARGET_COL]]
print(f'\n피처 수: {len(FEAT_COLS)}')
print(f'피처 목록: {FEAT_COLS}')

In [ ]:
# 타겟 변환
# TG: sqrt 변환 (높은 가격 범위 정규화)
# 비TG: log1p 변환

Xy_all['price_transformed'] = Xy_all.apply(
    lambda r: np.sqrt(r[TARGET_COL]) if r.get('item_TG', False) else np.log1p(r[TARGET_COL]),
    axis=1
)

print('타겟 변환 완료')
print(f'  TG  : sqrt 변환  → 범위 [{Xy_all[Xy_all["item_TG"]==True]["price_transformed"].min():.1f}, '
      f'{Xy_all[Xy_all["item_TG"]==True]["price_transformed"].max():.1f}]')
print(f'  비TG: log1p 변환 → 범위 [{Xy_all[Xy_all["item_TG"]==False]["price_transformed"].min():.2f}, '
      f'{Xy_all[Xy_all["item_TG"]==False]["price_transformed"].max():.2f}]')

---
## 5. 학습/검증 분리 (시간순)

In [ ]:
# 시간순 8:2 분리
TARGET_TRANSFORMED = 'price_transformed'
Xy_sorted = Xy_all.sort_values('year_month').reset_index(drop=True)

split_idx = int(len(Xy_sorted) * 0.8)
train_df = Xy_sorted.iloc[:split_idx].copy()
val_df   = Xy_sorted.iloc[split_idx:].copy()

X_tr = train_df[FEAT_COLS].values
y_tr = train_df[TARGET_TRANSFORMED].values
X_vl = val_df[FEAT_COLS].values
y_vl = val_df[TARGET_TRANSFORMED].values

# 검증용 원본 가격 보존
y_vl_orig   = val_df[TARGET_COL].values
y_vl_is_tg  = val_df.get('item_TG', pd.Series([False]*len(val_df))).values

print(f'학습: {X_tr.shape},  검증: {X_vl.shape}')

---
## 6. 피처 스케일링

In [ ]:
scaler = StandardScaler()
X_tr_scaled = scaler.fit_transform(X_tr)
X_vl_scaled = scaler.transform(X_vl)

# 테스트 피처도 스케일링 준비
test_feat_cols = [c for c in FEAT_COLS if c in test_enc.columns]
# 누락 컬럼은 0으로 채움
for col in FEAT_COLS:
    if col not in test_enc.columns:
        test_enc[col] = 0
X_test = test_enc[FEAT_COLS].values
X_test_scaled = scaler.transform(X_test)

print(f'스케일링 완료')
print(f'X_tr_scaled: mean={X_tr_scaled.mean():.4f}, std={X_tr_scaled.std():.4f}')

---
## 7. Keras DNN 모델 설계

```
Input (N피처) → Dense(256) → BN → ReLU → Dropout(0.3)
             → Dense(128) → BN → ReLU → Dropout(0.2)
             → Dense(64)  → BN → ReLU → Dropout(0.1)
             → Dense(32)  → BN → ReLU
             → Dense(1)   (Output)
```

In [ ]:
def build_model(input_dim, learning_rate=1e-3):
    l2 = regularizers.l2(1e-4)
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(256, kernel_regularizer=l2),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Dropout(0.3),
        layers.Dense(128, kernel_regularizer=l2),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Dropout(0.2),
        layers.Dense(64, kernel_regularizer=l2),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Dropout(0.1),
        layers.Dense(32, kernel_regularizer=l2),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Dense(1)
    ], name='jeju_dnn_v1')
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='mae',
        metrics=['mse']
    )
    return model


model = build_model(len(FEAT_COLS))
model.summary()

---
## 8. 학습

In [ ]:
os.makedirs('./models', exist_ok=True)
os.makedirs('./results', exist_ok=True)

MODEL_PATH = './models/jeju_dnn_v1.0.1.keras'

cb_list = [
    callbacks.EarlyStopping(monitor='val_loss', patience=20,
                            restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                patience=8, min_lr=1e-6, verbose=1),
    callbacks.ModelCheckpoint(filepath=MODEL_PATH, monitor='val_loss',
                              save_best_only=True, verbose=0)
]

history = model.fit(
    X_tr_scaled, y_tr,
    validation_data=(X_vl_scaled, y_vl),
    epochs=200,
    batch_size=512,
    callbacks=cb_list,
    verbose=1
)

best_epoch = np.argmin(history.history['val_loss'])
print(f'\nBest Epoch: {best_epoch}, Best Val MAE: {min(history.history["val_loss"]):.4f}')

---
## 9. 학습 곡선 시각화

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['loss'],     label='Train MAE', color='steelblue')
axes[0].plot(history.history['val_loss'], label='Val MAE',   color='darkorange', linestyle='--')
axes[0].axvline(best_epoch, color='red', linestyle=':', alpha=0.7, label=f'Best({best_epoch})')
axes[0].set_title('MAE 학습 곡선', fontsize=13)
axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['mse'],     label='Train MSE', color='steelblue')
axes[1].plot(history.history['val_mse'], label='Val MSE',   color='darkorange', linestyle='--')
axes[1].set_title('MSE 학습 곡선', fontsize=13)
axes[1].set_xlabel('Epoch'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle('Keras DNN v1.0.1 학습 곡선', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('./results/training_curve_v1.0.1.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 10. 모델 평가

In [ ]:
y_pred_transformed = model.predict(X_vl_scaled, verbose=0).flatten()

# 역변환: TG=제곱, 비TG=expm1
y_pred_orig = np.where(
    y_vl_is_tg.astype(bool),
    np.power(np.clip(y_pred_transformed, 0, None), 2),
    np.expm1(y_pred_transformed)
)
y_pred_orig = np.clip(y_pred_orig, 0, None)

mae  = mean_absolute_error(y_vl_orig, y_pred_orig)
rmse = np.sqrt(mean_squared_error(y_vl_orig, y_pred_orig))
r2   = r2_score(y_vl_orig, y_pred_orig)
mape = np.mean(np.abs((y_vl_orig - y_pred_orig) / (y_vl_orig + 1e-8))) * 100

print('=' * 50)
print('  Keras DNN v1.0.1 검증 성능')
print('=' * 50)
print(f'  MAE  : {mae:>10,.2f} 원/kg')
print(f'  RMSE : {rmse:>10,.2f} 원/kg')
print(f'  R²   : {r2:>10.4f}')
print(f'  MAPE : {mape:>10.2f} %')
print('=' * 50)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

max_v = max(y_vl_orig.max(), y_pred_orig.max())
axes[0].scatter(y_vl_orig, y_pred_orig, alpha=0.3, s=8, color='steelblue')
axes[0].plot([0, max_v], [0, max_v], 'r--', lw=2)
axes[0].set_title(f'실제 vs 예측  R²={r2:.4f}', fontsize=12)
axes[0].set_xlabel('실제'); axes[0].set_ylabel('예측'); axes[0].grid(True, alpha=0.3)

res = y_vl_orig - y_pred_orig
axes[1].hist(res, bins=50, color='darkorange', edgecolor='white', alpha=0.8)
axes[1].axvline(0, color='red', lw=2, linestyle='--')
axes[1].set_title(f'잔차 분포  mean={res.mean():.1f}', fontsize=12)
axes[1].set_xlabel('잔차'); axes[1].grid(True, alpha=0.3)

n = min(300, len(y_vl_orig))
axes[2].plot(y_vl_orig[:n],   label='실제', lw=1.2)
axes[2].plot(y_pred_orig[:n], label='예측', lw=1.2, linestyle='--')
axes[2].set_title(f'실제 vs 예측 (처음 {n}개)', fontsize=12)
axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.suptitle('Keras DNN v1.0.1 검증 결과', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('./results/eval_plots_v1.0.1.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 11. 테스트 예측 및 후처리

> **후처리 (DACON 1위 참고)**: 품목별 3월 최솟값 미만 예측 → 0으로 처리

In [ ]:
# 테스트 예측
test_pred_transformed = model.predict(X_test_scaled, verbose=0).flatten()

# TG 여부 확인
test_is_tg = test_enc.get('item_TG', pd.Series([False]*len(test_enc))).values.astype(bool)

# 역변환
test_pred_orig = np.where(
    test_is_tg,
    np.power(np.clip(test_pred_transformed, 0, None), 2),
    np.expm1(test_pred_transformed)
)
test_pred_orig = np.clip(test_pred_orig, 0, None)

# 후처리: 품목별 3월 최솟값 기준 하한 처리
# (DACON 1위: 너무 낮은 예측값은 0으로 처리)
test_enc_result = test_enc.copy()
test_enc_result['answer'] = test_pred_orig
test_enc_result['item'] = test_pre['item'].values  # 원본 item 복원

min_thresholds = {'TG': 400, 'CB': 50, 'RD': 10, 'CR': 150, 'BC': 100}
for item, thr in min_thresholds.items():
    mask = (test_enc_result['item'] == item) & (test_enc_result['answer'] < thr)
    test_enc_result.loc[mask, 'answer'] = 0
    n = mask.sum()
    if n > 0:
        print(f'{item}: {n}개 → 0 처리 (임계값: {thr})')

print(f'\n예측 통계:')
print(test_enc_result.groupby('item')['answer'].agg(['mean','min','max']).round(1))

---
## 12. 제출 파일 생성

In [ ]:
pred_df = test_enc_result[['ID', 'answer']].copy()

# sample_submission 순서에 맞게 정렬
result = sub[['ID']].merge(pred_df, on='ID', how='left')
result['answer'] = result['answer'].fillna(0)

SUBMISSION_PATH = './results/submission_v1.0.0.csv'
result.to_csv(SUBMISSION_PATH, index=False, encoding='utf-8-sig')

print(f'저장 완료: {SUBMISSION_PATH}')
print(f'행 수: {len(result)}, 결측치: {result["answer"].isnull().sum()}')
result.head(10)

---
## 13. 결과 요약

In [ ]:
print('=' * 60)
print('  Keras DNN v1.0.0 최종 결과')
print('=' * 60)
print(f'  입력 피처 수 : {len(FEAT_COLS)}개')
print(f'  학습 데이터  : {X_tr.shape[0]:,}행')
print(f'  검증 데이터  : {X_vl.shape[0]:,}행')
print(f'  Best Epoch   : {best_epoch}')
print()
print('  [검증 성능]')
print(f'  MAE  = {mae:>10,.2f} 원/kg')
print(f'  RMSE = {rmse:>10,.2f} 원/kg')
print(f'  R²   = {r2:>10.4f}')
print(f'  MAPE = {mape:>10.2f} %')
print()
print(f'  제출 파일: {SUBMISSION_PATH}')
print(f'  모델 파일: {MODEL_PATH}')
print('=' * 60)

### 다음 버전 개선 방향

| 버전 | 개선 내용 |
|------|-----------|
| **v1.1.0** | Embedding 레이어로 item/corp/loc 처리 |
| **v1.2.0** | Residual Connection 추가 |
| **v2.0.0** | LSTM/GRU 시계열 모델 |
| **v3.0.0** | DNN + CatBoost + XGBoost 앙상블 |